# 🔬 RoboQuest 2026 — 上級モード（コードを改造しよう）

このノートブックでは **Python コードを直接編集** して、ロボットの動きをより深くカスタマイズできます。

**できること:**
- 独自の報酬関数を追加する
- 観測空間を拡張する（ロボットに新しい感覚を与える）
- PPO のハイパーパラメータをチューニングする
- ニューラルネットワークのアーキテクチャを変える

---

In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os

# ── 1. GPU/CPU を自動検出してレンダリングバックエンドを設定 ──────────────────
#    mujoco を import する前に MUJOCO_GL を設定する必要がある
_has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0

if _has_gpu:
    # GPU あり → NVIDIA EGL（ハードウェアアクセラレーション）
    NVIDIA_ICD_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    if not os.path.exists(NVIDIA_ICD_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_PATH), exist_ok=True)
        with open(NVIDIA_ICD_PATH, 'w') as _f:
            _f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\n')
    os.environ['MUJOCO_GL'] = 'egl'
    print('🖥️  GPU 検出: EGL レンダリングを使用します')
else:
    # CPU のみ → OSMesa（ソフトウェアレンダリング、GPU 不要）
    subprocess.run('apt-get install -y -q libosmesa6', shell=True, check=False)
    os.environ['MUJOCO_GL'] = 'osmesa'
    print('💻  CPU モード: OSMesa レンダリングを使用します')

# ffmpeg（動画保存に必要）
subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ── 2. Python ライブラリのインストール ──────────────────────────────────────
print('ライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco', 'gymnasium', 'stable-baselines3', 'mediapy', 'tqdm', 'pandas', 'matplotlib'],
    check=True)

# ── 3. リポジトリのクローン ─────────────────────────────────────────────────
if not os.path.exists('/content/RoboQuest2026'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/tadryo/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# ── 4. Go2 モデルのダウンロード ─────────────────────────────────────────────
print('Go2 ロボットのモデルをダウンロード中...')
subprocess.run([sys.executable, 'scripts/download_models.py'], check=True)

print('\n✅ セットアップ完了！次のセルへ進んでください。')


## 第1章: 環境を理解しよう

### 観測空間（ロボットが「見ている」情報）

ロボットは以下の情報を観測として受け取ります（合計 52 次元）:

| インデックス | 内容 | 次元数 |
|-------------|------|-------|
| [0:3]       | 胴体の位置 (x, y, z) | 3 |
| [3:7]       | 胴体の向き（四元数） | 4 |
| [7:10]      | 胴体の速度 (vx, vy, vz) | 3 |
| [10:13]     | 胴体の角速度 | 3 |
| [13:25]     | 関節角度（立ち姿勢からの差） | 12 |
| [25:37]     | 関節の角速度 | 12 |
| [37:49]     | 前回の行動 | 12 |
| [49:51]     | 鬼への相対位置 (dx, dy) | 2 |
| [51]        | 鬼までの距離 | 1 |

In [ ]:
# 環境を確認してみよう
from roboquest.envs.go2_tag_env import Go2TagEnv

env = Go2TagEnv()
obs, info = env.reset()

print(f'観測空間の形: {env.observation_space.shape}')
print(f'行動空間の形: {env.action_space.shape}')
print(f'\n最初の観測値（一部）:')
print(f'  胴体位置: {obs[0:3]}')
print(f'  鬼への相対位置: {obs[49:51]}')
print(f'  鬼までの距離: {obs[51]:.2f} m')

## 第2章: 報酬関数を改造しよう

報酬関数はロボットが「何を良しとするか」を決める核心部分です。
以下のコードを自由に変更してみてください！

In [ ]:
# ===================================================
# ここを自由に改造しよう！
# ===================================================

from roboquest.utils.reward_utils import RewardConfig

# 独自の報酬設定
my_reward_config = RewardConfig(
    distance_weight=1.5,   # 鬼から離れる重み（大きいほど逃げ優先）
    survival_weight=0.2,   # 生存ボーナス
    control_weight=0.03,   # エネルギー効率
    forward_weight=0.8,    # 前進重み
    fall_penalty=15.0,     # 転倒ペナルティ
    tag_penalty=25.0,      # タグされたペナルティ
    boundary_penalty=3.0,  # アリーナ境界ペナルティ
)

print('報酬設定を確認:')
for field, value in my_reward_config.__dict__.items():
    print(f'  {field}: {value}')

### 独自の報酬関数を追加してみよう

Go2TagEnv を継承して、新しい報酬項を追加できます。

In [ ]:
import numpy as np
from roboquest.envs.go2_tag_env import Go2TagEnv

class MyCustomEnv(Go2TagEnv):
    """独自の報酬関数を持つカスタム環境"""

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)

        # === ここに独自の報酬を追加しよう ===

        # 例1: 素早く逃げるボーナス（速度が大きいほど高報酬）
        speed = float(np.linalg.norm(self.data.qvel[:2]))  # xy 方向の速度
        reward += 0.1 * speed

        # 例2: 方向ボーナス（鬼から遠ざかる方向に進んでいればボーナス）
        oni_dir = self._oni_pos - self.robot_xy
        if np.linalg.norm(oni_dir) > 0:
            robot_vel = self.data.qvel[:2]
            flee_dir = -oni_dir / np.linalg.norm(oni_dir)  # 鬼と逆方向
            alignment = float(np.dot(robot_vel, flee_dir))  # 内積
            reward += 0.05 * max(0, alignment)  # 逃げ方向に動いていたらボーナス

        # 例3: アリーナ中央から離れすぎないペナルティ（自由に追加）
        # dist_from_center = float(np.linalg.norm(self.robot_xy))
        # if dist_from_center > 4.0:
        #     reward -= 0.1

        return obs, reward, terminated, truncated, info

# テスト
env = MyCustomEnv(reward_config=my_reward_config)
obs, _ = env.reset()
obs, r, done, trunc, info = env.step(env.action_space.sample())
print(f'カスタム報酬テスト: reward = {r:.4f}')
print('✅ カスタム環境が動作しています')

## 第3章: PPO ハイパーパラメータを探求しよう

PPO（Proximal Policy Optimization）の主なパラメータ:

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor

# ===================================================
# ここを自由に改造しよう！
# ===================================================

HYPERPARAMS = {
    'learning_rate': 3e-4,     # 学習率（小さいほど慎重に学習）
    'n_steps': 2048,           # 1回の更新で集めるステップ数
    'batch_size': 512,         # ミニバッチサイズ
    'n_epochs': 10,            # 1回のデータで何回更新するか
    'gamma': 0.99,             # 割引率（将来の報酬をどれだけ重視するか）
    'gae_lambda': 0.95,        # GAE lambda（アドバンテージ推定の精度）
    'clip_range': 0.2,         # クリッピング範囲（PPO の核心パラメータ）
    'ent_coef': 0.01,          # エントロピー係数（探索の多様性）
}

NET_ARCH = [256, 256, 128]    # ニューラルネットの構造（各層のユニット数）

TOTAL_TIMESTEPS = 500_000
NUM_ENVS = 4

# 環境作成
def make_env(rank=0):
    def _init():
        env = MyCustomEnv(reward_config=my_reward_config)
        return Monitor(env, f'/tmp/monitor_adv_{rank}')
    return _init

train_env = SubprocVecEnv([make_env(i) for i in range(NUM_ENVS)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True)

model = PPO(
    'MlpPolicy',
    train_env,
    policy_kwargs={'net_arch': NET_ARCH},
    **HYPERPARAMS,
    verbose=1,
)

print(f'モデルのパラメータ数: {sum(p.numel() for p in model.policy.parameters()):,}')
print(f'ネットワーク構造: {NET_ARCH}')
print('学習準備完了')

In [ ]:
# 学習実行
from stable_baselines3.common.callbacks import CheckpointCallback

print(f'=== 学習開始 (上級モード) ===')
print(f'合計ステップ数: {TOTAL_TIMESTEPS:,}')

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=CheckpointCallback(save_freq=100_000, save_path='/tmp/checkpoints'),
    progress_bar=True,
)

model.save('/tmp/flee_advanced')
train_env.save('/tmp/vec_normalize_adv.pkl')
print('✅ 学習完了')

## 第4章: 結果を確認して大会準備

In [ ]:
# 学習曲線の表示
from roboquest.utils.visualization import plot_training_curve
plot_training_curve('/tmp', title='上級モード学習曲線')

In [ ]:
# 鬼ごっこ動画の録画と表示
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from roboquest.envs.go2_tag_env import Go2TagEnv

loaded = PPO.load('/tmp/flee_advanced')
env = Go2TagEnv(render_mode='rgb_array')
renderer = mujoco.Renderer(env.model, height=480, width=640)

frames = []
obs, _ = env.reset()
distances = []

for _ in range(800):
    action, _ = loaded.predict(obs, deterministic=True)
    obs, r, terminated, truncated, info = env.step(action)
    distances.append(info.get('oni_distance', 0))
    renderer.update_scene(env.data)
    frames.append(renderer.render().copy())
    if terminated or truncated:
        break

renderer.close()
env.close()

try:
    import mediapy as media
    media.show_video(frames, fps=30)
except ImportError:
    from matplotlib import animation
    from IPython.display import HTML, display
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('off')
    img = ax.imshow(frames[0])
    ani = animation.FuncAnimation(
        fig, lambda f: [img.set_array(f)], frames=frames, interval=33, blit=True)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

print(f'生存ステップ: {len(frames)}')
print(f'平均距離: {np.mean(distances):.2f} m')

In [ ]:
# Google Drive に保存して大会提出
from google.colab import drive
drive.mount('/content/drive')

import os, yaml, shutil

# Colab ヘッドレス環境用: EGL オフスクリーンレンダリングを有効化
os.environ["MUJOCO_GL"] = "egl"

team_name = input('チーム名を入力: ')  # Tier2 は直接入力
SAVE_DIR = f'/content/drive/MyDrive/RoboQuest2026/{team_name}'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(f'{SAVE_DIR}/flee_final')
train_env.save(f'{SAVE_DIR}/vec_normalize.pkl')

params = {
    'team_name': team_name,
    'mode': 'tier2_advanced',
    'hyperparams': HYPERPARAMS,
    'net_arch': NET_ARCH,
    'reward_config': my_reward_config.__dict__,
    'avg_distance': float(np.mean(distances)),
}
with open(f'{SAVE_DIR}/params.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(params, f, allow_unicode=True)

print(f'🎉 提出完了！保存先: {SAVE_DIR}')